In [1]:
import pandas as pd
import numpy as np

recs_df    = pd.read_csv('../outputs/recommendations.csv')
candidates = pd.read_csv('../data/processed/candidates_clean.csv')
jobs       = pd.read_csv('../data/processed/jobs_clean.csv')

# ══════════════════════════════════════════════════════════════════
# METRIC 1 — Average top-1 similarity score
# ══════════════════════════════════════════════════════════════════
top1 = recs_df[recs_df['Rank'] == 1]
print(f"Average top-1 skill similarity: {top1['Skill_Similarity'].mean():.4f}")
print(f"Average top-1 final score:      {top1['Final_Score'].mean():.4f}")

# ══════════════════════════════════════════════════════════════════
# METRIC 2 — Skill overlap ratio (how many required skills does the
#             candidate actually have, out of total required skills)
# ══════════════════════════════════════════════════════════════════
def skill_overlap(cand_skills_str, job_skills_str):
    cand = set(s.strip() for s in str(cand_skills_str).split(','))
    job  = set(s.strip() for s in str(job_skills_str).split(','))
    if not job:
        return 0.0
    return len(cand & job) / len(job)

recs_df['Skill_Overlap_Ratio'] = recs_df.apply(
    lambda r: skill_overlap(r['Candidate_Skills'], r['Required_Skills']), axis=1
)

print(f"\nAverage skill overlap ratio (all recs): "
      f"{recs_df['Skill_Overlap_Ratio'].mean():.4f}")
print(f"Average skill overlap ratio (top-1 only): "
      f"{recs_df[recs_df['Rank']==1]['Skill_Overlap_Ratio'].mean():.4f}")

# ══════════════════════════════════════════════════════════════════
# METRIC 3 — Coverage: % of jobs that appear in at least 1 recommendation
# ══════════════════════════════════════════════════════════════════
covered_jobs   = recs_df['Job_Title'].nunique()
total_jobs     = len(jobs)
coverage       = covered_jobs / total_jobs * 100
print(f"\nJob coverage: {covered_jobs}/{total_jobs} = {coverage:.1f}%")

# ══════════════════════════════════════════════════════════════════
# METRIC 4 — Experience level match rate
# ══════════════════════════════════════════════════════════════════
recs_df['Level_Match'] = recs_df['Candidate_Exp_Level'] == recs_df['Job_Exp_Level']
match_rate = recs_df[recs_df['Rank']==1]['Level_Match'].mean() * 100
print(f"Experience level match rate (top-1): {match_rate:.1f}%")

# ══════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════
summary = {
    'Metric': [
        'Avg top-1 skill similarity',
        'Avg top-1 final score',
        'Avg skill overlap ratio (top-1)',
        'Job catalog coverage',
        'Experience level match rate (top-1)',
    ],
    'Value': [
        f"{top1['Skill_Similarity'].mean():.4f}",
        f"{top1['Final_Score'].mean():.4f}",
        f"{recs_df[recs_df['Rank']==1]['Skill_Overlap_Ratio'].mean():.4f}",
        f"{coverage:.1f}%",
        f"{match_rate:.1f}%",
    ]
}
print("\n── Evaluation Summary ──")
display(pd.DataFrame(summary))

recs_df.to_csv('../outputs/recommendations_with_metrics.csv', index=False)

Average top-1 skill similarity: 0.2548
Average top-1 final score:      0.2316

Average skill overlap ratio (all recs): 0.7742
Average skill overlap ratio (top-1 only): 0.7742

Job coverage: 243/50000 = 0.5%
Experience level match rate (top-1): 100.0%

── Evaluation Summary ──


,Metric,Value
0,Avg top-1 skill similarity,0.2548
1,Avg top-1 final score,0.2316
2,Avg skill overlap ratio (top-1),0.7742
3,Job catalog coverage,0.5%
4,Experience level match rate (top-1),100.0%
